# 04 Paretro: Pareto-based Lab Candidate Selection

This notebook implements a multi-objective candidate selection workflow
for generated NylC variants.

The filename keeps the requested `Paretro` spelling, but the method is
Pareto-based selection.

Goals:

- train a conservative descriptor-based ANOVA-GP ensemble from the
  current lab data;
- reconstruct mutations for generated BoltzGen candidates;
- score candidates by predicted PA6 activity, model uncertainty,
  structural quality, diversity, mutation burden and liability risk;
- compute separate exploitation, exploration and balanced Pareto fronts;
- select a diverse laboratory test panel plus controls;
- export all tables needed for the next design-build-test-learn round.

## Scientific caution

This workflow should be interpreted as **candidate prioritization**, not
as calibrated absolute prediction. The ANOVA-GP model is trained on a
small four-position lab dataset. Generated variants may contain
mutations outside those modeled positions, so predictions are best read
as pocket-projection scores. The Pareto workflow explicitly keeps
structure, diversity and mutation burden visible instead of collapsing
everything into a single optimized acquisition score.

In [ ]:
# Run once in a fresh environment if peptides.py is missing.
# In the ba_nylc cluster environment this is usually already installed.
%pip install peptides

In [ ]:
from dataclasses import dataclass
from itertools import product
from pathlib import Path
import math
import re
import sys
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.spatial.distance import cdist
from scipy.stats import pearsonr, spearmanr
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler

import peptides
from peptides import Peptide

pd.set_option("display.max_columns", 160)
pd.set_option("display.width", 180)
rng = np.random.default_rng(42)


def find_project_root(start=None):
    start = Path.cwd() if start is None else Path(start).resolve()
    for candidate in [start] + list(start.parents):
        if (candidate / "matrices" / "NylC_Puetz_raw_data.CSV").exists():
            return candidate
    if start.name == "Notebooks":
        return start.parent
    raise FileNotFoundError("Could not locate project root.")


PROJECT_ROOT = find_project_root()
DATA_PATH = PROJECT_ROOT / "matrices" / "NylC_Puetz_raw_data.CSV"
WT_FASTA = PROJECT_ROOT / "inputs" / "variant_fastas" / "WT.fasta"
BOLTZ_DIR = PROJECT_ROOT / "inputs" / "boltz_variants"
OUT_DIR = PROJECT_ROOT / "results" / "paretro_selection"
OUT_DIR.mkdir(parents=True, exist_ok=True)

print("Python:", sys.version.split()[0])
print("peptides.py:", getattr(peptides, "__version__", "unknown"))
print("Project root:", PROJECT_ROOT)
print("Output:", OUT_DIR)

## Configuration

Adjust `LAB_PANEL_SIZE` and `MAX_MUTATIONS_HARD` before a real lab
round. The default panel uses 17 generated variants plus 3 controls.

In [ ]:
LAB_PANEL_SIZE = 20
N_CONTROLS = 3
N_GENERATED_TO_SELECT = LAB_PANEL_SIZE - N_CONTROLS

MAX_MUTATIONS_HARD = 8
MAX_MUTATIONS_PREFERRED = 5
MAX_PER_MUTATION_SIGNATURE = 3
MIN_QUALITY_SCORE = 0.0

# Default final-model ensemble. These settings come from the reviewed
# ANOVA-GP sensitivity analysis and are deliberately simple.
ENSEMBLE_DESCRIPTOR_SETS = ["physical", "z_scales", "vhse", "kidera", "pcp", "atchley"]
DEFAULT_MODEL_SETTINGS = {
    "physical": {"lengthscale": 0.2, "sigma_main": 0.5, "sigma_epi": 4.0, "sigma_noise": 0.5},
    "z_scales": {"lengthscale": 0.5, "sigma_main": 0.5, "sigma_epi": 4.0, "sigma_noise": 0.5},
    "vhse": {"lengthscale": 0.5, "sigma_main": 0.5, "sigma_epi": 4.0, "sigma_noise": 0.5},
    "kidera": {"lengthscale": 0.2, "sigma_main": 0.5, "sigma_epi": 4.0, "sigma_noise": 0.5},
    "pcp": {"lengthscale": 0.2, "sigma_main": 0.5, "sigma_epi": 4.0, "sigma_noise": 0.5},
    "atchley": {"lengthscale": 0.2, "sigma_main": 0.5, "sigma_epi": 4.0, "sigma_noise": 0.5},
}

print("Generated candidates to select:", N_GENERATED_TO_SELECT)
print("Descriptor ensemble:", ENSEMBLE_DESCRIPTOR_SETS)

## Load lab data and WT sequence

In [ ]:
def read_fasta_sequence(path):
    return "".join(
        line.strip()
        for line in Path(path).read_text().splitlines()
        if line.strip() and not line.startswith(">")
    )


WT_SEQUENCE = read_fasta_sequence(WT_FASTA)
print("WT length:", len(WT_SEQUENCE))

df_raw = pd.read_csv(DATA_PATH, sep=";", decimal=",")
activity_rep_cols = [col for col in df_raw.columns if col.startswith("activity_pa6_")]
tm_rep_cols = [col for col in df_raw.columns if col.startswith("tm_celsius_")]

df_lab = df_raw.copy()
df_lab[activity_rep_cols] = df_lab[activity_rep_cols].apply(pd.to_numeric, errors="coerce")
df_lab[tm_rep_cols] = df_lab[tm_rep_cols].apply(pd.to_numeric, errors="coerce")
df_lab["activity_n"] = df_lab[activity_rep_cols].count(axis=1)
df_lab["activity_pa6"] = df_lab[activity_rep_cols].mean(axis=1, skipna=True)
df_lab["activity_sd"] = df_lab[activity_rep_cols].std(axis=1, skipna=True, ddof=1)
df_lab["activity_sem"] = df_lab["activity_sd"] / np.sqrt(df_lab["activity_n"])
df_lab["tm_celsius"] = df_lab[tm_rep_cols].mean(axis=1, skipna=True)

fallback_sem = df_lab.loc[
    df_lab["activity_sem"].notna() & (df_lab["activity_sem"] > 0),
    "activity_sem",
].median()
df_lab["activity_sem_for_gp"] = df_lab["activity_sem"].fillna(fallback_sem)

display(df_lab[["variant_id", "mutations", "activity_n", "activity_pa6", "activity_sem_for_gp", "tm_celsius"]].head(12))
print("Lab variants:", len(df_lab))

## Descriptor-based ANOVA-GP model

This is a compact self-contained copy of the reviewed model logic. It
is used here for final candidate scoring, not for claiming a calibrated
absolute activity predictor.

In [ ]:
CANONICAL_AA = tuple("ACDEFGHIKLMNPQRSTVWY")
WT_POCKET = {99: "D", 134: "F", 304: "D", 330: "R"}
POSITIONS = tuple(WT_POCKET.keys())
TARGET_POSITIONS = set(POSITIONS)

DESCRIPTOR_SOURCES = {
    "kidera": {"method": "kidera_factors", "columns": [f"KF{i}" for i in range(1, 11)]},
    "vhse": {"method": "vhse_scales", "columns": [f"VHSE{i}" for i in range(1, 9)]},
    "z_scales": {"method": "z_scales", "columns": [f"Z{i}" for i in range(1, 6)]},
    "atchley": {"method": "atchley_factors", "columns": [f"AF{i}" for i in range(1, 6)]},
    "pcp": {"method": "pcp_descriptors", "columns": [f"PCP{i}" for i in range(1, 6)]},
    "physical": {"method": "physical_descriptors", "columns": ["PD1", "PD2"]},
}


@dataclass(frozen=True)
class DescriptorSource:
    name: str
    method: str
    raw_table: pd.DataFrame
    scaled_table: pd.DataFrame


def residue_descriptor_table(source_name, standardize=True):
    cfg = DESCRIPTOR_SOURCES[source_name]
    rows = {}
    for aa in CANONICAL_AA:
        values = np.asarray(getattr(Peptide(aa), cfg["method"])(), dtype=float)
        rows[aa] = values
    raw = pd.DataFrame.from_dict(rows, orient="index", columns=cfg["columns"]).sort_index()
    if standardize:
        scaled = pd.DataFrame(StandardScaler().fit_transform(raw), index=raw.index, columns=raw.columns)
    else:
        scaled = raw.copy()
    return DescriptorSource(source_name, cfg["method"], raw, scaled)


DESCRIPTOR_CACHE = {}


def get_descriptor_source(source_name):
    if source_name not in DESCRIPTOR_CACHE:
        DESCRIPTOR_CACHE[source_name] = residue_descriptor_table(source_name, standardize=True)
    return DESCRIPTOR_CACHE[source_name]


MUTATION_PATTERN = re.compile(r"([A-Z])(\d+)([A-Z])")


def parse_mutation_string(mutation_string):
    if pd.isna(mutation_string) or str(mutation_string).strip().lower() in {"", "wt", "wildtype", "wild type", "nan"}:
        return []
    return [(wt, int(pos), mut) for wt, pos, mut in MUTATION_PATTERN.findall(str(mutation_string).upper())]


def mutation_list_to_string(mutations):
    if not mutations:
        return ""
    return ";".join(f"{wt}{pos}{mut}" for wt, pos, mut in sorted(mutations, key=lambda x: x[1]))


def pocket_from_mutations(mutations):
    pocket = dict(WT_POCKET)
    for wt, pos, mut in mutations:
        if pos in pocket:
            pocket[pos] = mut
    return pocket


def mutation_positions(mutations):
    return {pos for _, pos, _ in mutations}


def mutation_signature(positions):
    if not positions:
        return "WT"
    return "+".join(str(pos) for pos in sorted(positions))


def add_mutation_features(df, mutation_col="mutations"):
    out = df.copy()
    parsed = out[mutation_col].apply(parse_mutation_string)
    out["parsed_mutations"] = parsed
    out["mutation_set"] = parsed.apply(lambda muts: frozenset(f"{wt}{pos}{mut}" for wt, pos, mut in muts))
    out["n_mutations_reconstructed"] = parsed.apply(len)
    out["mutation_positions"] = parsed.apply(mutation_positions)
    out["mutation_signature"] = out["mutation_positions"].apply(mutation_signature)
    out["pocket"] = parsed.apply(pocket_from_mutations)
    out["aa_tuple"] = out["pocket"].apply(lambda pocket: tuple(pocket[pos] for pos in POSITIONS))
    out["only_gp_target_positions"] = out["mutation_positions"].apply(lambda pos: pos.issubset(TARGET_POSITIONS))
    out["n_external_mutations"] = out["parsed_mutations"].apply(lambda muts: sum(pos not in TARGET_POSITIONS for _, pos, _ in muts))
    return out


df_lab_model = add_mutation_features(df_lab, "mutations")
df_train_gp = df_lab_model[df_lab_model["only_gp_target_positions"]].copy().reset_index(drop=True)
print("GP training variants:", len(df_train_gp), "of", len(df_lab_model))
display(df_train_gp[["variant_id", "mutations", "aa_tuple", "activity_pa6"]].head())

In [ ]:
def aa_descriptor_vector(aa, source):
    return source.scaled_table.loc[str(aa).upper()].to_numpy(dtype=float)


def pockets_to_position_descriptor_arrays(pockets, source, positions=POSITIONS):
    return {
        pos: np.vstack([aa_descriptor_vector(pocket[pos], source) for pocket in pockets])
        for pos in positions
    }


def rbf_kernel_from_descriptors(X1, X2=None, lengthscale=1.0):
    if X2 is None:
        X2 = X1
    X1 = np.asarray(X1, dtype=float)
    X2 = np.asarray(X2, dtype=float)
    squared_distances = ((X1[:, None, :] - X2[None, :, :]) ** 2).sum(axis=2)
    return np.exp(-squared_distances / (2.0 * float(lengthscale) ** 2))


def build_position_kernels(position_arrays, lengthscale=1.0):
    return {
        pos: rbf_kernel_from_descriptors(values, lengthscale=lengthscale)
        for pos, values in position_arrays.items()
    }


def build_position_cross_kernels(train_pockets, test_pockets, source, lengthscale=1.0):
    train_arrays = pockets_to_position_descriptor_arrays(train_pockets, source)
    test_arrays = pockets_to_position_descriptor_arrays(test_pockets, source)
    return {
        pos: rbf_kernel_from_descriptors(test_arrays[pos], X2=train_arrays[pos], lengthscale=lengthscale)
        for pos in POSITIONS
    }


def build_main_kernel(position_kernels):
    return np.mean(np.stack(list(position_kernels.values()), axis=0), axis=0)


def build_epistasis_kernel(position_kernels):
    positions = list(position_kernels)
    products = []
    for i, pos_a in enumerate(positions):
        for pos_b in positions[i + 1:]:
            products.append(position_kernels[pos_a] * position_kernels[pos_b])
    return np.mean(np.stack(products, axis=0), axis=0)


def build_total_anova_kernel(position_kernels, sigma_main=0.5, sigma_epi=4.0):
    return float(sigma_main) ** 2 * build_main_kernel(position_kernels) + float(sigma_epi) ** 2 * build_epistasis_kernel(position_kernels)


def build_total_cross_kernel(train_pockets, test_pockets, source, lengthscale=1.0, sigma_main=0.5, sigma_epi=4.0):
    cross = build_position_cross_kernels(train_pockets, test_pockets, source, lengthscale=lengthscale)
    return float(sigma_main) ** 2 * build_main_kernel(cross) + float(sigma_epi) ** 2 * build_epistasis_kernel(cross)


def standardize_target(y):
    y = np.asarray(y, dtype=float)
    y_mean = float(np.mean(y))
    y_std = float(np.std(y, ddof=1))
    return (y - y_mean) / y_std, y_mean, y_std


def fit_gp_from_kernel(K_total, y, sem=None, sigma_noise=0.5, jitter=1e-8):
    y_scaled, y_mean, y_std = standardize_target(y)
    n = len(y_scaled)
    if sem is None:
        sem_scaled = np.zeros(n)
    else:
        sem_scaled = np.nan_to_num(np.asarray(sem, dtype=float) / y_std, nan=0.0)
    K_y = (
        K_total
        + np.diag(sem_scaled ** 2)
        + float(sigma_noise) ** 2 * np.eye(n)
        + float(jitter) * np.eye(n)
    )
    L = np.linalg.cholesky(K_y)
    alpha = np.linalg.solve(L.T, np.linalg.solve(L, y_scaled))
    return {
        "K_total": K_total,
        "K_y": K_y,
        "L": L,
        "alpha": alpha,
        "y_mean": y_mean,
        "y_std": y_std,
        "sigma_noise": float(sigma_noise),
    }


def gp_predict_from_fit(gp_fit, K_test_train, K_test_diag):
    mean_scaled = K_test_train @ gp_fit["alpha"]
    v = np.linalg.solve(gp_fit["L"], K_test_train.T)
    var_latent_scaled = np.maximum(K_test_diag - np.sum(v ** 2, axis=0), 0.0)
    var_observed_scaled = var_latent_scaled + gp_fit["sigma_noise"] ** 2
    return {
        "mean": gp_fit["y_mean"] + gp_fit["y_std"] * mean_scaled,
        "std_observed": gp_fit["y_std"] * np.sqrt(var_observed_scaled),
    }


def fit_final_gp_model(df_train, source_name, settings):
    source = get_descriptor_source(source_name)
    train_pockets = df_train["pocket"].tolist()
    arrays = pockets_to_position_descriptor_arrays(train_pockets, source)
    kernels = build_position_kernels(arrays, lengthscale=settings["lengthscale"])
    K_train = build_total_anova_kernel(kernels, sigma_main=settings["sigma_main"], sigma_epi=settings["sigma_epi"])
    fit = fit_gp_from_kernel(
        K_train,
        y=df_train["activity_pa6"].to_numpy(dtype=float),
        sem=df_train["activity_sem_for_gp"].to_numpy(dtype=float),
        sigma_noise=settings["sigma_noise"],
    )
    return {"source_name": source_name, "source": source, "settings": settings, "train_pockets": train_pockets, "fit": fit}


def predict_with_final_gp_model(model, candidate_pockets):
    settings = model["settings"]
    K_test_train = build_total_cross_kernel(
        model["train_pockets"],
        candidate_pockets,
        source=model["source"],
        lengthscale=settings["lengthscale"],
        sigma_main=settings["sigma_main"],
        sigma_epi=settings["sigma_epi"],
    )
    K_test_diag = np.full(len(candidate_pockets), float(settings["sigma_main"]) ** 2 + float(settings["sigma_epi"]) ** 2)
    return gp_predict_from_fit(model["fit"], K_test_train, K_test_diag)


final_models = {
    source_name: fit_final_gp_model(df_train_gp, source_name, DEFAULT_MODEL_SETTINGS[source_name])
    for source_name in ENSEMBLE_DESCRIPTOR_SETS
}
print("Fitted final descriptor models:", list(final_models))

## Load generated BoltzGen candidates

Raw BoltzGen tables do not always include a clean mutation column.
Mutations are reconstructed by aligning each designed fragment to the
WT sequence with a simple offset scan. `X` is treated as unknown and
ignored during comparison.

In [ ]:
def best_fragment_offset(fragment, reference):
    fragment = str(fragment)
    if len(fragment) > len(reference):
        return None, np.nan
    best = None
    for offset in range(len(reference) - len(fragment) + 1):
        matches = 0
        compared = 0
        for i, aa in enumerate(fragment):
            if aa == "X" or pd.isna(aa):
                continue
            compared += 1
            if reference[offset + i] == aa:
                matches += 1
        score = matches / compared if compared else np.nan
        if best is None or score > best[0]:
            best = (score, offset)
    return best[1], best[0]


def mutations_from_fragment(fragment, reference):
    fragment = str(fragment)
    offset, identity = best_fragment_offset(fragment, reference)
    if offset is None:
        return [], offset, identity
    muts = []
    for i, aa in enumerate(fragment):
        if aa == "X":
            continue
        pos = offset + i + 1
        wt = reference[offset + i]
        if aa != wt:
            muts.append((wt, pos, aa))
    return muts, offset, identity


def load_boltzgen_candidates(boltz_dir=BOLTZ_DIR):
    rows = []
    for csv_file in sorted(Path(boltz_dir).glob("all_designs_metrics_*.csv")):
        run_name = csv_file.stem.replace("all_designs_metrics_", "")
        df_run = pd.read_csv(csv_file)
        df_run["boltzgen_run"] = run_name
        df_run["source_file"] = csv_file.name
        rows.append(df_run)
    if not rows:
        raise FileNotFoundError(f"No all_designs_metrics_*.csv files found in {boltz_dir}")
    candidates = pd.concat(rows, ignore_index=True, sort=False)
    candidates["candidate_id"] = candidates["boltzgen_run"].astype(str) + "__" + candidates["id"].astype(str)
    return candidates


candidates_raw = load_boltzgen_candidates()
reconstructed = candidates_raw["designed_chain_sequence"].apply(lambda seq: mutations_from_fragment(seq, WT_SEQUENCE))
candidates_raw["parsed_mutations"] = reconstructed.apply(lambda item: item[0])
candidates_raw["fragment_offset_0based"] = reconstructed.apply(lambda item: item[1])
candidates_raw["fragment_identity_to_wt"] = reconstructed.apply(lambda item: item[2])
candidates_raw["mutations"] = candidates_raw["parsed_mutations"].apply(mutation_list_to_string)

candidates = add_mutation_features(candidates_raw, mutation_col="mutations")

tested_mutation_sets = set(df_lab_model["mutation_set"])
candidates["already_tested"] = candidates["mutation_set"].isin(tested_mutation_sets)
candidates["n_mutations"] = candidates["parsed_mutations"].apply(len)

print("Raw generated candidates:", len(candidates))
print("Already tested mutation sets:", candidates["already_tested"].sum())
display(candidates[["candidate_id", "boltzgen_run", "fragment_identity_to_wt", "mutations", "n_mutations", "aa_tuple"]].head(10))

## Predict generated candidates with descriptor ensemble

In [ ]:
candidate_pockets = candidates["pocket"].tolist()
ensemble_prediction_tables = []

for source_name, model in final_models.items():
    pred = predict_with_final_gp_model(model, candidate_pockets)
    ensemble_prediction_tables.append(pd.DataFrame({
        "candidate_id": candidates["candidate_id"].values,
        f"pred_activity_{source_name}": pred["mean"],
        f"pred_std_{source_name}": pred["std_observed"],
    }))

predictions = ensemble_prediction_tables[0]
for table in ensemble_prediction_tables[1:]:
    predictions = predictions.merge(table, on="candidate_id", how="left")

pred_cols = [c for c in predictions.columns if c.startswith("pred_activity_")]
std_cols = [c for c in predictions.columns if c.startswith("pred_std_")]

predictions["predicted_activity_gp_mean"] = predictions[pred_cols].mean(axis=1)
predictions["predicted_activity_gp_sd_between_descriptors"] = predictions[pred_cols].std(axis=1)
predictions["predicted_activity_gp_std_mean"] = predictions[std_cols].mean(axis=1)
predictions["model_uncertainty"] = np.sqrt(
    predictions["predicted_activity_gp_sd_between_descriptors"].fillna(0) ** 2
    + predictions["predicted_activity_gp_std_mean"].fillna(0) ** 2
)

candidates = candidates.merge(predictions, on="candidate_id", how="left")
display(candidates[["candidate_id", "mutations", "predicted_activity_gp_mean", "model_uncertainty"]].head())

## Scoring helpers: structure, novelty, diversity and filters

In [ ]:
def robust_z(series, higher_is_better=True):
    s = pd.to_numeric(series, errors="coerce")
    med = s.median()
    mad = (s - med).abs().median()
    if not np.isfinite(mad) or mad == 0:
        std = s.std(ddof=0)
        z = (s - med) / std if std and np.isfinite(std) else pd.Series(0.0, index=s.index)
    else:
        z = 0.6745 * (s - med) / mad
    z = z.replace([np.inf, -np.inf], np.nan).fillna(0.0)
    return z if higher_is_better else -z


def safe_numeric_col(df, col, default=np.nan):
    if col in df.columns:
        return pd.to_numeric(df[col], errors="coerce")
    return pd.Series(default, index=df.index)


candidates["structure_score"] = (
    robust_z(safe_numeric_col(candidates, "quality_score"), True)
    + robust_z(safe_numeric_col(candidates, "design_to_target_iptm"), True)
    + robust_z(safe_numeric_col(candidates, "design_ptm"), True)
    + robust_z(safe_numeric_col(candidates, "filter_rmsd"), False)
    + robust_z(safe_numeric_col(candidates, "min_design_to_target_pae"), False)
    + robust_z(safe_numeric_col(candidates, "liability_score"), False)
)

candidates["mutation_burden"] = candidates["n_mutations"]
candidates["mutation_count_penalty"] = np.maximum(0, candidates["n_mutations"] - MAX_MUTATIONS_PREFERRED)
candidates["liability_score_numeric"] = safe_numeric_col(candidates, "liability_score", default=0).fillna(0)
candidates["quality_score_numeric"] = safe_numeric_col(candidates, "quality_score", default=0).fillna(0)

def jaccard_distance(set_a, set_b):
    set_a = set(set_a)
    set_b = set(set_b)
    if not set_a and not set_b:
        return 0.0
    return 1.0 - len(set_a & set_b) / len(set_a | set_b)


tested_sets = list(df_lab_model["mutation_set"])
candidates["diversity_from_tested"] = candidates["mutation_set"].apply(
    lambda s: min(jaccard_distance(s, tested) for tested in tested_sets)
)

candidates["has_x_filter_fail"] = (
    candidates["has_x"].astype(bool)
    if "has_x" in candidates.columns
    else False
)
candidates["hard_filter_pass"] = (
    (~candidates["already_tested"])
    & (candidates["n_mutations"] <= MAX_MUTATIONS_HARD)
    & (~candidates["has_x_filter_fail"])
    & (candidates["quality_score_numeric"] >= MIN_QUALITY_SCORE)
)

display(candidates[[
    "candidate_id", "mutations", "n_mutations", "hard_filter_pass",
    "predicted_activity_gp_mean", "model_uncertainty", "structure_score",
    "diversity_from_tested", "quality_score_numeric", "liability_score_numeric",
]].head(12))
print("Candidates passing hard filters:", candidates["hard_filter_pass"].sum(), "of", len(candidates))

## Pareto fronts

Three fronts are computed:

- **exploitation:** high predicted activity and structure quality, low
  mutation burden and liability;
- **exploration:** high uncertainty and diversity while maintaining
  acceptable predicted activity and structure;
- **balanced:** activity, structure and diversity together.

In [ ]:
def pareto_front(df, maximize=None, minimize=None):
    maximize = list(maximize or [])
    minimize = list(minimize or [])
    cols = maximize + minimize
    work = df.copy().reset_index(drop=True)
    values = work[cols].apply(pd.to_numeric, errors="coerce")
    values = values.fillna(values.median(numeric_only=True)).fillna(0.0)
    oriented = values.copy()
    for col in minimize:
        oriented[col] = -oriented[col]
    arr = oriented.to_numpy(dtype=float)
    n = len(work)
    is_efficient = np.ones(n, dtype=bool)
    for i in range(n):
        if not is_efficient[i]:
            continue
        dominates_i = np.all(arr >= arr[i], axis=1) & np.any(arr > arr[i], axis=1)
        if np.any(dominates_i):
            is_efficient[i] = False
    return work[is_efficient].copy()


candidate_pool = candidates[candidates["hard_filter_pass"]].copy()

exploitation_front = pareto_front(
    candidate_pool,
    maximize=["predicted_activity_gp_mean", "structure_score", "quality_score_numeric"],
    minimize=["mutation_burden", "liability_score_numeric", "filter_rmsd"],
)
exploitation_front["pareto_group"] = "exploitation"

exploration_front = pareto_front(
    candidate_pool,
    maximize=["model_uncertainty", "diversity_from_tested", "predicted_activity_gp_mean", "structure_score"],
    minimize=["mutation_burden", "liability_score_numeric"],
)
exploration_front["pareto_group"] = "exploration"

balanced_front = pareto_front(
    candidate_pool,
    maximize=["predicted_activity_gp_mean", "structure_score", "diversity_from_tested"],
    minimize=["mutation_burden", "liability_score_numeric", "filter_rmsd"],
)
balanced_front["pareto_group"] = "balanced"

print("candidate_pool:", len(candidate_pool))
print("exploitation_front:", len(exploitation_front))
print("exploration_front:", len(exploration_front))
print("balanced_front:", len(balanced_front))

display(exploitation_front[["candidate_id", "mutations", "predicted_activity_gp_mean", "structure_score", "n_mutations"]].head(10))

## Greedy diverse panel selection

Pareto can still return too many candidates. The panel selection below
uses quotas and a greedy diversity rule so one mutation family cannot
dominate the full plate.

In [ ]:
def panel_distance(row, selected_rows):
    if not selected_rows:
        return 1.0
    current = row["mutation_set"]
    return min(jaccard_distance(current, selected["mutation_set"]) for selected in selected_rows)


def prepare_front_for_selection(front, primary_cols):
    out = front.copy()
    score = pd.Series(0.0, index=out.index)
    for col, higher in primary_cols:
        score += robust_z(out[col], higher_is_better=higher)
    out["within_front_score"] = score
    return out.sort_values("within_front_score", ascending=False)


def greedy_select(front, quota, selected_rows, label):
    selected_ids = {row["candidate_id"] for row in selected_rows}
    signature_counts = {}
    for row in selected_rows:
        signature_counts[row["mutation_signature"]] = signature_counts.get(row["mutation_signature"], 0) + 1

    chosen = []
    front = front.copy()
    for _, row in front.iterrows():
        if len(chosen) >= quota:
            break
        if row["candidate_id"] in selected_ids:
            continue
        signature = row["mutation_signature"]
        if signature_counts.get(signature, 0) >= MAX_PER_MUTATION_SIGNATURE:
            continue
        row = row.copy()
        row["selection_bucket"] = label
        row["panel_diversity_distance"] = panel_distance(row, selected_rows + chosen)
        chosen.append(row)
        selected_ids.add(row["candidate_id"])
        signature_counts[signature] = signature_counts.get(signature, 0) + 1
    return chosen


exploitation_ranked = prepare_front_for_selection(
    exploitation_front,
    [
        ("predicted_activity_gp_mean", True),
        ("structure_score", True),
        ("mutation_burden", False),
        ("liability_score_numeric", False),
    ],
)
exploration_ranked = prepare_front_for_selection(
    exploration_front,
    [
        ("model_uncertainty", True),
        ("diversity_from_tested", True),
        ("structure_score", True),
        ("mutation_burden", False),
    ],
)
balanced_ranked = prepare_front_for_selection(
    balanced_front,
    [
        ("predicted_activity_gp_mean", True),
        ("diversity_from_tested", True),
        ("structure_score", True),
        ("mutation_burden", False),
    ],
)

selected_rows = []
quotas = {
    "exploitation": max(1, int(round(N_GENERATED_TO_SELECT * 0.45))),
    "exploration": max(1, int(round(N_GENERATED_TO_SELECT * 0.30))),
}
quotas["balanced"] = max(0, N_GENERATED_TO_SELECT - sum(quotas.values()))

selected_rows += greedy_select(exploitation_ranked, quotas["exploitation"], selected_rows, "exploitation")
selected_rows += greedy_select(exploration_ranked, quotas["exploration"], selected_rows, "exploration")
selected_rows += greedy_select(balanced_ranked, quotas["balanced"], selected_rows, "balanced")

# Fill remaining slots from the union of all fronts if quotas were too restrictive.
if len(selected_rows) < N_GENERATED_TO_SELECT:
    union_front = pd.concat([exploitation_ranked, exploration_ranked, balanced_ranked], ignore_index=True)
    union_front = union_front.drop_duplicates("candidate_id")
    union_front = prepare_front_for_selection(
        union_front,
        [
            ("predicted_activity_gp_mean", True),
            ("structure_score", True),
            ("diversity_from_tested", True),
            ("mutation_burden", False),
        ],
    )
    selected_rows += greedy_select(union_front, N_GENERATED_TO_SELECT - len(selected_rows), selected_rows, "fill_from_pareto_union")

lab_panel_generated = pd.DataFrame(selected_rows)
lab_panel_generated = lab_panel_generated.head(N_GENERATED_TO_SELECT).copy()
lab_panel_generated["panel_rank"] = np.arange(1, len(lab_panel_generated) + 1)

panel_cols = [
    "panel_rank", "selection_bucket", "candidate_id", "boltzgen_run", "final_rank",
    "mutations", "n_mutations", "mutation_signature", "predicted_activity_gp_mean",
    "model_uncertainty", "structure_score", "diversity_from_tested",
    "quality_score_numeric", "design_to_target_iptm", "design_ptm",
    "filter_rmsd", "liability_score_numeric", "panel_diversity_distance",
]
panel_cols = [col for col in panel_cols if col in lab_panel_generated.columns]
display(lab_panel_generated[panel_cols])

## Suggested controls

Controls are chosen from already characterized variants: WT, a strong
positive control and a middle/low activity control. Adjust manually
based on assay plate layout and material availability.

In [ ]:
controls = []
if "WT" in set(df_lab["variant_id"]):
    controls.append(df_lab[df_lab["variant_id"] == "WT"].iloc[0])

positive = df_lab.sort_values("activity_pa6", ascending=False).iloc[0]
if positive["variant_id"] not in [row["variant_id"] for row in controls]:
    controls.append(positive)

median_idx = (df_lab["activity_pa6"] - df_lab["activity_pa6"].median()).abs().idxmin()
middle = df_lab.loc[median_idx]
if middle["variant_id"] not in [row["variant_id"] for row in controls]:
    controls.append(middle)

controls_df = pd.DataFrame(controls).head(N_CONTROLS).copy()
controls_df["selection_bucket"] = "control"
controls_df["panel_rank"] = np.arange(len(lab_panel_generated) + 1, len(lab_panel_generated) + 1 + len(controls_df))

display(controls_df[["panel_rank", "selection_bucket", "variant_id", "mutations", "activity_pa6", "tm_celsius"]])

## Export

In [ ]:
candidates.to_csv(OUT_DIR / "all_generated_candidates_scored.csv", index=False)
candidate_pool.to_csv(OUT_DIR / "candidate_pool_after_hard_filters.csv", index=False)
exploitation_front.to_csv(OUT_DIR / "pareto_front_exploitation.csv", index=False)
exploration_front.to_csv(OUT_DIR / "pareto_front_exploration.csv", index=False)
balanced_front.to_csv(OUT_DIR / "pareto_front_balanced.csv", index=False)
lab_panel_generated.to_csv(OUT_DIR / "lab_test_panel_generated_pareto.csv", index=False)
controls_df.to_csv(OUT_DIR / "lab_test_panel_controls.csv", index=False)

panel_summary = {
    "n_raw_candidates": len(candidates),
    "n_candidate_pool": len(candidate_pool),
    "n_exploitation_front": len(exploitation_front),
    "n_exploration_front": len(exploration_front),
    "n_balanced_front": len(balanced_front),
    "n_generated_selected": len(lab_panel_generated),
    "n_controls": len(controls_df),
}
pd.DataFrame([panel_summary]).to_csv(OUT_DIR / "paretro_selection_summary.csv", index=False)

print("Exported to:", OUT_DIR)
for key, value in panel_summary.items():
    print(f"{key}: {value}")

## How to update after a lab round

After testing a selected panel:

1. Append the measured variants to `matrices/NylC_Puetz_raw_data.CSV`
   or to a round-specific lab-data file with the same replicate columns.
2. Add `source_round` and `selection_bucket` columns if possible. They
   are not required by this notebook, but they make later analysis much
   easier.
3. Re-run this notebook. The GP ensemble, Pareto fronts and panel will
   update automatically.
4. Compare the newly measured activity distribution by selection bucket
   to learn whether exploitation, exploration or balanced picks were
   most useful.